# 29. Coronal Mass Ejections: Observations — Implementation / 구현

**Paper**: Webb and Howard (2012), *Living Rev. Solar Phys.*, 9, 3.

## Purpose / 목적

**English.** This notebook reproduces key observational analyses of CMEs from the Webb and Howard (2012) review:

1. Height-time diagrams and kinematic fits (linear / quadratic).
2. CME speed distributions (log-normal).
3. Mass estimate from Thomson-scattering brightness (simplified).
4. Halo vs limb CME identification by angular width.
5. Projection effects on measured speed.
6. Simple cone model of CME geometry.

**한국어.** 이 노트북은 Webb and Howard(2012) 리뷰의 주요 관측 분석을 재현한다:

1. 고도-시간 다이어그램과 운동학 적합(선형 / 2차).
2. CME 속도 분포(로그정규).
3. Thomson 산란 밝기로부터의 질량 추정(단순 모델).
4. 겉보기 각폭에 의한 헤일로 vs 변두리 CME 판별.
5. 투영 효과에 의한 측정 속도 오차.
6. CME 기하에 대한 단순 원뿔 모델.

In [ ]:
# Standard imports / 표준 임포트
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

rng = np.random.default_rng(seed=42)

# Physical constants / 물리 상수
R_SUN_KM = 6.957e5        # Solar radius in km
M_PROTON_G = 1.6726e-24   # Proton mass in g
AU_KM = 1.496e8           # 1 AU in km

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("Solar radius:", R_SUN_KM, "km")
print("Proton mass:", M_PROTON_G, "g")

---
## 1. Height-time Diagram / 고도-시간 다이어그램

**English.** CMEs are tracked by measuring the leading-edge height at successive times. The height-time profile reveals kinematics:

- Linear fit: constant-speed CME.
- Quadratic fit: accelerating/decelerating CME.

We generate synthetic data with known v and a, add noise, and fit both models.

**한국어.** CME는 코로나그래프 영상에서 시간에 따른 선단 높이로 추적한다. 선형 적합은 일정 속도 CME, 2차 적합은 가속/감속 CME. 알려진 v, a로 합성 데이터를 만들고 잡음을 더한 뒤 두 모델을 적합한다.

In [ ]:
def generate_synthetic_ht(t_min, v0_kms, a_ms2, h0_rsun, noise_rsun=0.1):
    """Generate synthetic CME height-time data.

    Args:
        t_min: Time in minutes.
        v0_kms: Initial speed in km/s.
        a_ms2: Acceleration in m/s^2.
        h0_rsun: Starting height in R_sun.
        noise_rsun: Gaussian noise standard deviation (R_sun).

    Returns:
        Height in R_sun.
    """
    t_s = t_min * 60.0
    displacement_km = v0_kms * t_s + 0.5 * (a_ms2 / 1000.0) * t_s ** 2
    h_rsun = h0_rsun + displacement_km / R_SUN_KM
    noise = rng.normal(0, noise_rsun, size=h_rsun.shape)
    return h_rsun + noise


t_minutes = np.linspace(0, 120, 15)
h_gradual = generate_synthetic_ht(t_minutes, 200.0, 8.0, 3.0)
h_impulsive = generate_synthetic_ht(t_minutes, 1500.0, -15.0, 3.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(t_minutes, h_gradual, "o", color="tab:blue", label="Synthetic data")
axes[0].set_title("Gradual CME (v0=200 km/s, a=+8 m/s^2) / 점진형 CME")
axes[0].set_xlabel("Time (min) / 시간")
axes[0].set_ylabel("Height (R_sun) / 고도")
axes[0].legend()

axes[1].plot(t_minutes, h_impulsive, "s", color="tab:red", label="Synthetic data")
axes[1].set_title("Impulsive CME (v0=1500 km/s, a=-15 m/s^2) / 충격형 CME")
axes[1].set_xlabel("Time (min) / 시간")
axes[1].set_ylabel("Height (R_sun) / 고도")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
def linear_model(t_s, h0_km, v_kms):
    """Linear (constant-speed) height-time model.

    Args:
        t_s: Time in seconds.
        h0_km: Initial height in km.
        v_kms: Constant speed in km/s.

    Returns:
        Height in km.
    """
    return h0_km + v_kms * t_s


def quadratic_model(t_s, h0_km, v0_kms, a_ms2):
    """Quadratic height-time model with constant acceleration.

    Args:
        t_s: Time in seconds.
        h0_km: Initial height in km.
        v0_kms: Initial speed in km/s.
        a_ms2: Acceleration in m/s^2.

    Returns:
        Height in km.
    """
    return h0_km + v0_kms * t_s + 0.5 * (a_ms2 / 1000.0) * t_s ** 2


def fit_kinematics(t_minutes, h_rsun):
    """Fit linear and quadratic models to a height-time track.

    Args:
        t_minutes: Times in minutes.
        h_rsun: Heights in R_sun.

    Returns:
        Dict with fit parameters and RMS residuals.
    """
    t_s = t_minutes * 60.0
    h_km = h_rsun * R_SUN_KM
    p_lin, _ = curve_fit(linear_model, t_s, h_km, p0=[h_km[0], 400.0])
    p_quad, _ = curve_fit(quadratic_model, t_s, h_km, p0=[h_km[0], 400.0, 0.0])
    res_lin = np.sqrt(np.mean((linear_model(t_s, *p_lin) - h_km) ** 2)) / R_SUN_KM
    res_quad = np.sqrt(np.mean((quadratic_model(t_s, *p_quad) - h_km) ** 2)) / R_SUN_KM
    return {"linear": p_lin, "quadratic": p_quad, "rms_linear": res_lin, "rms_quad": res_quad}


fit_grad = fit_kinematics(t_minutes, h_gradual)
fit_impuls = fit_kinematics(t_minutes, h_impulsive)

print("Gradual CME fit:")
v = fit_grad["linear"][1]
a = fit_grad["quadratic"][2]
rms = fit_grad["rms_linear"]
rmsq = fit_grad["rms_quad"]
print("  Linear    : v = {:.1f} km/s, rms = {:.3f} Rsun".format(v, rms))
print("  Quadratic : a = {:.2f} m/s^2, rms = {:.3f} Rsun".format(a, rmsq))
print()
print("Impulsive CME fit:")
v = fit_impuls["linear"][1]
a = fit_impuls["quadratic"][2]
rms = fit_impuls["rms_linear"]
rmsq = fit_impuls["rms_quad"]
print("  Linear    : v = {:.1f} km/s, rms = {:.3f} Rsun".format(v, rms))
print("  Quadratic : a = {:.2f} m/s^2, rms = {:.3f} Rsun".format(a, rmsq))

In [ ]:
t_dense_min = np.linspace(0, 120, 200)
t_dense_s = t_dense_min * 60

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
data_list = [h_gradual, h_impulsive]
fit_list = [fit_grad, fit_impuls]
labels = ["Gradual", "Impulsive"]
colors = ["tab:blue", "tab:red"]
for ax, data, fit, label, color in zip(axes, data_list, fit_list, labels, colors):
    h_lin = linear_model(t_dense_s, *fit["linear"]) / R_SUN_KM
    h_quad = quadratic_model(t_dense_s, *fit["quadratic"]) / R_SUN_KM
    ax.plot(t_minutes, data, "o", color=color, label="Data")
    ax.plot(t_dense_min, h_lin, "--", color="black", label="Linear v = {:.0f} km/s".format(fit["linear"][1]))
    ax.plot(t_dense_min, h_quad, "-", color=color, alpha=0.6, label="Quadratic a = {:.1f} m/s^2".format(fit["quadratic"][2]))
    ax.set_xlabel("Time (min) / 시간")
    ax.set_ylabel("Height (R_sun) / 고도")
    ax.set_title(label + " CME H-T fit / 고도-시간 적합")
    ax.legend()
plt.tight_layout()
plt.show()

**Interpretation / 해석.**

**English.** For gradual (slow, accelerating) CMEs the quadratic fit captures the acceleration clearly. For impulsive CMEs — which Webb and Howard section 2.5 note are nearly constant-speed above 2 R_sun — both linear and quadratic fits give similar speeds. This matches the review observation that **only 17 percent of LASCO CMEs show measurable acceleration** out to 30 R_sun.

**한국어.** 점진형(느리고 가속되는) CME는 2차 적합이 가속을 명확히 포착한다. 충격형 CME는 2 R_sun 이상에서 거의 일정 속도이며 선형·2차 적합 모두 비슷한 속도를 준다. 이는 **LASCO CME 중 30 R_sun까지 측정 가능한 가속을 보이는 것은 17 %에 불과**하다는 리뷰 관측과 일치한다.

---
## 2. CME Speed Distribution (Log-normal) / CME 속도 분포

**English.** Yurchyshyn et al. (2005) showed LASCO CME speed distributions are **log-normal**, implying many simultaneous or sequential processes. Figure 12 of Webb and Howard gives an average of ~466 km/s for all CMEs (10 896 events). We sample a log-normal distribution and compare.

**한국어.** Yurchyshyn et al.(2005)은 LASCO CME 속도 분포가 **로그정규**임을 보였으며, 이는 동시 다수 과정 또는 순차적 일련의 과정이 작용함을 시사한다. Webb and Howard 그림 12의 LASCO 전체 평균은 10 896건 기준 ~466 km/s다.

In [ ]:
def sample_cme_speeds(n, mu_ln_v=np.log(400.0), sigma_ln_v=0.6):
    """Draw CME speeds from a log-normal distribution.

    Default parameters reproduce the LASCO all-CME distribution
    (mean around 466 km/s, long tail to over 2500 km/s).

    Args:
        n: Number of samples.
        mu_ln_v: Mean of ln(v) with v in km/s.
        sigma_ln_v: Std of ln(v).

    Returns:
        Array of CME speeds in km/s.
    """
    return rng.lognormal(mean=mu_ln_v, sigma=sigma_ln_v, size=n)


speeds = sample_cme_speeds(n=10896)

print("Mean speed   : {:.1f} km/s  (LASCO: ~466)".format(speeds.mean()))
print("Median speed : {:.1f} km/s".format(np.median(speeds)))
print("Max speed    : {:.1f} km/s".format(speeds.max()))
print("Fraction > 1000 km/s: {:.1f} %".format((speeds > 1000).mean() * 100))
print("Fraction > 2000 km/s: {:.1f} %".format((speeds > 2000).mean() * 100))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(speeds, bins=np.linspace(0, 2500, 50), color="gray", edgecolor="k")
axes[0].axvline(speeds.mean(), color="red", linestyle="--", label="Mean {:.0f} km/s".format(speeds.mean()))
axes[0].set_xlabel("Speed (km/s) / 속도")
axes[0].set_ylabel("Number / 개수")
axes[0].set_title("Linear scale / 선형 스케일")
axes[0].set_xlim(0, 2500)
axes[0].legend()

axes[1].hist(speeds, bins=np.logspace(np.log10(20), np.log10(5000), 50), color="gray", edgecolor="k")
axes[1].set_xscale("log")
axes[1].set_xlabel("Speed (km/s, log) / 속도 (로그)")
axes[1].set_ylabel("Number / 개수")
axes[1].set_title("Log scale: log-normal shape / 로그 스케일")
plt.tight_layout()
plt.show()

---
## 3. CME Mass from Thomson Scattering / Thomson 산란 질량

**English.** The observed brightness excess is related to electron column density via the Thomson-scattering geometric factor G(chi). Under the sky-plane assumption (chi=90), G is maximized. Simplified inversion: given brightness excess and area, return mass assuming mu=1.92 (H + 10% He).

**한국어.** 밝기 초과와 전자 기둥밀도는 Thomson 기하 인자 G(chi)로 연결된다. Sky plane 가정(chi=90)에서 G가 최대. 단순 역변환으로 밝기와 면적에서 질량 계산(mu=1.92, 수소 + 10 % 헬륨).

In [ ]:
SIGMA_T_CM2 = 6.6524e-25
MU_CME = 1.92


def thomson_geometric_factor(chi_deg):
    """Simplified Thomson scattering geometric factor.

    Uses the dominant sin^2(chi) dependence, normalized to 1 at 90 degrees.

    Args:
        chi_deg: Scattering angle in degrees.

    Returns:
        Geometric factor (0 to 1).
    """
    chi_rad = np.deg2rad(chi_deg)
    return np.sin(chi_rad) ** 2


def cme_mass_from_brightness(brightness_excess, area_cm2, chi_deg=90.0):
    """Estimate total CME mass from Thomson-scattering brightness excess.

    Calibrated so a canonical LASCO CME yields ~1e15 g.

    Args:
        brightness_excess: Brightness excess (fraction of B_sun).
        area_cm2: Projected CME area in cm^2.
        chi_deg: Mean scattering angle (deg).

    Returns:
        Total CME mass in grams.
    """
    g_factor = thomson_geometric_factor(chi_deg)
    brightness_scale = 5e-14
    n_e_col = brightness_excess / (SIGMA_T_CM2 * g_factor * brightness_scale)
    mass_g = n_e_col * area_cm2 * MU_CME * M_PROTON_G
    return mass_g


area_test_cm2 = 3.0 * (R_SUN_KM * 1e5) ** 2
b_excess = 5e-15
mass_g = cme_mass_from_brightness(b_excess, area_test_cm2, 90.0)
print("Canonical LASCO CME mass: {:.2e} g = {:.2f} x 10^15 g".format(mass_g, mass_g / 1e15))
print("Compare to Table 1 LASCO average: 1.3 x 10^15 g")

In [ ]:
chi_deg_arr = np.linspace(30, 150, 100)
mass_vs_chi = np.array([cme_mass_from_brightness(b_excess, area_test_cm2, c) for c in chi_deg_arr])

plt.figure(figsize=(9, 5))
plt.plot(chi_deg_arr, mass_vs_chi / 1e15, "b-", linewidth=2)
plt.axvline(90, color="k", linestyle="--", alpha=0.5, label="Sky plane")
plt.axhline(mass_vs_chi.max() / 1e15, color="r", linestyle=":", label="Sky-plane mass: {:.2f} x 10^15 g".format(mass_vs_chi.max() / 1e15))
plt.xlabel("Scattering angle chi (deg) / 산란각")
plt.ylabel("Inferred mass (10^15 g) / 추정 질량")
plt.title("Thomson-scattering mass vs direction / Thomson 질량 - 방향")
plt.legend()
plt.show()

factor = thomson_geometric_factor(90) / thomson_geometric_factor(60)
print("When CME is 30 deg out of sky plane, inferred mass is higher by factor {:.2f}.".format(factor))
print("Vourlidas et al. (2010): masses underestimated by factor ~2 for Earth-directed CMEs.")

---
## 4. Halo vs Limb CME Classification / 헤일로 vs 변두리 분류

**English.** Webb and Howard section 2.3 classification by apparent angular width w:

- **Narrow**: w < 20 deg
- **Limb**: 20 <= w <= 120 deg
- **Partial halo**: 120 < w < 360 deg
- **Full halo**: w ~ 360 deg

Observed: ~10 % partial halo, ~4 % full halo.

**한국어.** Webb and Howard 2.3절은 겉보기 각폭 w에 따라 분류한다: narrow < 20, limb 20–120, partial halo 120–360, full halo 360. 관측 비율: 부분 헤일로 ~10 %, 완전 헤일로 ~4 %.

In [ ]:
def classify_cme_width(width_deg):
    """Classify a CME by apparent angular width.

    Args:
        width_deg: Apparent angular width (deg).

    Returns:
        One of narrow, limb, partial_halo, full_halo.
    """
    if width_deg < 20:
        return "narrow"
    if width_deg <= 120:
        return "limb"
    if width_deg < 330:
        return "partial_halo"
    return "full_halo"


def sample_cme_widths(n):
    """Sample CME widths matching Figure 12 of Webb and Howard.

    Produces average ~41 deg with ~10 percent partial halos and ~4 percent full halos.

    Args:
        n: Number of samples.

    Returns:
        Array of widths in degrees.
    """
    main = rng.lognormal(mean=np.log(35.0), sigma=0.7, size=n)
    is_halo = rng.random(n) < 0.10
    halo_width = rng.uniform(140, 360, size=n)
    widths = np.where(is_halo, halo_width, main)
    return np.clip(widths, 5, 360)


widths = sample_cme_widths(n=13125)
classes = np.array([classify_cme_width(w) for w in widths])

print("Total CMEs    :", len(widths))
print("Average width : {:.1f} deg (Fig 12: 41)".format(widths.mean()))
for cn in ["narrow", "limb", "partial_halo", "full_halo"]:
    print("  {:13}: {:5.1f} percent".format(cn, (classes == cn).mean() * 100))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(0, 360, 37)
ax.hist(widths, bins=bins, color="lightgray", edgecolor="black", label="All CMEs")
ax.axvline(20, color="blue", linestyle="--", label="narrow | limb")
ax.axvline(120, color="green", linestyle="--", label="limb | partial halo")
ax.axvline(330, color="red", linestyle="--", label="partial | full halo")
ax.axvline(widths.mean(), color="black", linestyle="-", linewidth=2, label="Mean {:.0f} deg".format(widths.mean()))
ax.set_xlabel("Apparent angular width (deg) / 겉보기 각폭")
ax.set_ylabel("Number / 개수")
ax.set_title("CME angular width distribution / CME 각폭 분포")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

---
## 5. Projection Effects / 투영 효과

**English.** Coronagraphs see only the sky-plane projection: v_measured = v_true * cos(phi), where phi is the angle from sky plane. Earth-directed halos (phi -> 90) are drastically underestimated. STEREO multi-viewpoint geometry breaks this degeneracy.

**한국어.** 코로나그래프는 sky-plane 투영만 본다: v_측정 = v_실제 * cos(phi). 지구 방향 헤일로(phi -> 90)는 심하게 과소평가. STEREO 다시점이 이 모호성 해결.

In [ ]:
def projected_speed(v_true_kms, phi_deg):
    """Sky-plane projection of a CME velocity.

    Args:
        v_true_kms: True CME speed in km/s.
        phi_deg: Angle from sky plane (deg). 90 = Earth-directed.

    Returns:
        Measured projected speed in km/s.
    """
    return v_true_kms * np.cos(np.deg2rad(phi_deg))


phi_deg_arr = np.linspace(0, 89, 90)
v_true = 1500.0
v_measured = projected_speed(v_true, phi_deg_arr)

plt.figure(figsize=(10, 5))
plt.plot(phi_deg_arr, v_measured, "b-", linewidth=2, label="Measured / 측정")
plt.axhline(v_true, color="red", linestyle="--", label="True = {:.0f} km/s".format(v_true))
plt.axvline(90, color="gray", linestyle=":", label="Earth-directed / 지구 방향")
plt.xlabel("Angle from sky plane (deg)")
plt.ylabel("Measured projected speed (km/s)")
plt.title("Projection effect on CME speed / 투영 효과")
plt.legend()
plt.show()

phi = 70.0
v_proj = projected_speed(v_true, phi)
t_true = AU_KM / v_true / 3600
t_proj = AU_KM / v_proj / 3600
print("CME at phi = {:.0f} deg from sky plane:".format(phi))
print("  True speed        : {:.0f} km/s".format(v_true))
print("  Projected speed   : {:.0f} km/s".format(v_proj))
print("  True arrival time : {:.1f} hr to 1 AU".format(t_true))
print("  Naive arrival time: {:.1f} hr (overestimate)".format(t_proj))

---
## 6. Simple Cone Model / 단순 원뿔 모델

**English.** A cone-model CME has apex at Sun center, axis along propagation direction n-hat, and half-angle omega. The apparent width seen from Earth is w_app = 2 * atan(tan(omega) / cos(phi)). When phi -> 90 deg (Earth-directed), w_app -> 360 deg — all Earth-directed CMEs look like halos regardless of true width.

**한국어.** 원뿔 모델 CME: 정점이 태양 중심, 축이 전파 방향 n-hat, 반각 omega. 겉보기 폭 w_app = 2*atan(tan(omega)/cos(phi)). phi -> 90°이면 고유 폭이 좁아도 w_app -> 360° — 모든 지구 방향 CME는 진짜 폭과 관계없이 헤일로처럼 보인다.

In [ ]:
def cone_apparent_width(intrinsic_half_angle_deg, phi_deg):
    """Apparent angular width of a cone-model CME.

    Args:
        intrinsic_half_angle_deg: Cone half-angle omega (deg).
        phi_deg: Angle between cone axis and sky plane (deg).

    Returns:
        Apparent full width (deg), capped at 360.
    """
    omega = np.deg2rad(intrinsic_half_angle_deg)
    phi = np.deg2rad(phi_deg)
    cos_phi = max(np.cos(phi), 1e-6)
    ratio = np.tan(omega) / cos_phi
    if ratio > 1e6:
        return 360.0
    return float(np.degrees(2 * np.arctan(ratio)))


phi_range = np.linspace(0, 85, 86)
intrinsic_omegas = [15, 30, 45, 60]

plt.figure(figsize=(10, 6))
for om in intrinsic_omegas:
    ws = [cone_apparent_width(om, p) for p in phi_range]
    plt.plot(phi_range, ws, label="omega = {} deg".format(om))
plt.axhline(120, color="gray", linestyle=":", label="Partial halo threshold")
plt.axhline(330, color="red", linestyle=":", label="Full halo threshold")
plt.xlabel("CME axis angle from sky plane (deg)")
plt.ylabel("Apparent width (deg)")
plt.title("Cone-model CME apparent width / 원뿔 모델 CME 겉보기 폭")
plt.legend()
plt.ylim(0, 360)
plt.show()

print("Narrow CME (omega=15 deg):")
for p in [0, 30, 60, 75, 85]:
    print("  phi = {:3d} deg -> apparent width {:6.1f} deg".format(p, cone_apparent_width(15, p)))

**Interpretation / 해석.**

**English.** The cone-model plot illustrates a critical observational reality: Earth-directed CMEs become halos regardless of their true intrinsic width. This is why halo-CME catalogs mix physically different events and why multi-viewpoint STEREO data are essential for disentangling propagation direction from intrinsic width.

**한국어.** 원뿔 모델 그래프는 핵심 관측 현실을 보여준다: 지구 방향 CME는 고유 폭과 관계없이 헤일로가 된다. 그래서 헤일로 CME 카탈로그는 물리적으로 다른 사건이 섞여 있으며, 전파 방향과 고유 폭을 분리하려면 STEREO의 다시점 자료가 필수적이다.

---
## 7. Summary / 요약

**English.** We have numerically reproduced and explored six fundamental aspects of CME observations from Webb and Howard (2012):

1. **Height-time kinematics** — linear and quadratic fits distinguish gradual (accelerating) from impulsive (nearly constant-speed) CMEs, consistent with the review finding that only 17 percent of LASCO CMEs show measurable acceleration above 2 R_sun.
2. **Speed distribution** — log-normal sampling reproduces the LASCO average of ~466 km/s with a long high-speed tail to >2000 km/s.
3. **Thomson-scattering mass** — a simplified inversion gives ~1e15 g for a canonical LASCO CME; scattering geometry demonstrates the systematic mass underestimation for Earth-directed events.
4. **Halo/limb classification** — the angular-width histogram reproduces ~10 percent partial halo / ~4 percent full halo fractions.
5. **Projection effects** — v_measured = v_true * cos(phi) quantifies why halos are slowed in single-viewpoint data.
6. **Cone geometry** — narrow CMEs look like halos when Earth-directed, motivating STEREO multi-viewpoint approach.

These simplified models illustrate the core quantitative concepts. Real CME analysis uses full Billings (1966) Thomson scattering theory, proper F+K corona subtraction, and polarization / stereoscopy.

**한국어.** Webb and Howard(2012)의 CME 관측 6가지 핵심을 수치적으로 재현·탐구했다:

1. **고도-시간 운동학** — 선형·2차 적합으로 점진형(가속)과 충격형(거의 일정속도) CME를 구별, 리뷰의 "2 R_sun 이상에서 LASCO CME의 17 %만 측정 가능한 가속" 관측과 일치.
2. **속도 분포** — 로그정규 표본으로 LASCO 평균 ~466 km/s와 >2000 km/s의 긴 고속 꼬리를 재현.
3. **Thomson 산란 질량** — 단순 역변환으로 대표 LASCO CME 질량 ~1e15 g; 산란 기하가 지구 방향 사건의 체계적 질량 과소추정을 보여줌.
4. **헤일로/변두리 분류** — 각폭 히스토그램이 부분 헤일로 ~10 %, 완전 헤일로 ~4 % 비율을 재현.
5. **투영 효과** — v_측정 = v_실제 * cos(phi)가 단일시점 자료에서 헤일로 속도가 작아지는 이유를 정량화.
6. **원뿔 기하** — 좁은 CME도 지구 방향이면 헤일로로 보임을 보여, STEREO 다시점 접근의 동기를 제공.

이 모델들은 단순화된 것이며, 실제 CME 분석은 완전한 Billings(1966) Thomson 산란 이론, 적절한 F+K 코로나 차감, 편광·입체시를 사용한다. 그러나 핵심 정량 개념을 잘 보여준다.